# 04 - CasADi Basics for MPC

This notebook introduces CasADi as a practical modeling and optimization tool for MPC.

The goal is not to build a reusable controller package. The goal is to see the basic CasADi workflow clearly: symbolic expressions, functions, derivatives, optimization variables, parameters, and a small receding-horizon MPC simulation.

## 1. Why CasADi?

CasADi is not MPC.

CasADi is a tool for symbolic modeling, automatic differentiation, and numerical optimization. We use it because MPC problems quickly become structured optimization problems.

In this notebook, every optimization problem is written directly in the notebook. That keeps the modeling steps visible.

## 2. Setup and version check

The first code cell uses notebook-safe plotting. It does not force a Matplotlib backend.

If CasADi is missing, this cell raises a clear error. There are no installation commands inside the notebook.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

try:
    import casadi as ca
except ImportError as exc:
    raise ImportError(
        "CasADi is required for this notebook. Install it in the active Python environment before running these examples."
    ) from exc

np.set_printoptions(precision=3, suppress=True)

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "lines.linewidth": 2.0,
})

p_opts = {"print_time": False}
s_opts = {"print_level": 0}

print("CasADi version:", ca.__version__)
print("Imports worked.")

## 3. Symbolic variables: SX, MX, DM

CasADi has several matrix types. For our purposes:

- `SX` is useful for simple symbolic scalar and matrix expressions.
- `MX` represents more general symbolic graphs, often useful for larger structured problems.
- `DM` is CasADi's numeric matrix type.

Keep shapes explicit. For control code, prefer column vectors for states and inputs.

In [ ]:
x = ca.SX.sym("x")
y = ca.SX.sym("y")
f_expr = x**2 + ca.sin(y)

print("SX expression:", f_expr)

f_eval = ca.Function("f_eval", [x, y], [f_expr])
value = f_eval(2.0, 0.5)
print("f(2.0, 0.5) =", float(value))

In [ ]:
z = ca.SX.sym("z", 2, 1)
z_expr = ca.vertcat(z[0] + z[1], z[0] - z[1])

z_fun = ca.Function("z_fun", [z], [z_expr])
z_value = z_fun(ca.DM([3.0, 1.0]))

print("z shape:", z.shape)
print("z_expr shape:", z_expr.shape)
print("z_fun([3, 1]) =")
print(np.array(z_value, dtype=float))

In [ ]:
mx = ca.MX.sym("mx", 2, 1)
mx_expr = ca.sumsqr(mx)
numeric_dm = ca.DM([[1.0], [2.0]])

print("MX expression:", mx_expr)
print("DM value:")
print(numeric_dm)
print("DM converted to NumPy:")
print(np.array(numeric_dm, dtype=float))

## 4. CasADi functions

A CasADi function packages symbolic inputs and symbolic outputs into something we can evaluate many times.

This is useful because the same symbolic expression can be evaluated with different numerical values without rebuilding the expression each time.

In [ ]:
x = ca.SX.sym("x")
y = ca.SX.sym("y")

symbolic_output = x**2 + ca.sin(y)
f = ca.Function("f", [x, y], [symbolic_output])

print("symbolic output:", symbolic_output)
print("function:", f)

result = f(2.0, np.pi / 2.0)
print("raw CasADi result:", result)
print("as float:", float(result))

## 5. Automatic differentiation

Automatic differentiation is one of the main reasons CasADi is useful for optimization and MPC.

We can ask CasADi for gradients, Jacobians, and Hessians of symbolic expressions.

In [ ]:
x = ca.SX.sym("x")
scalar_f = x**2 + ca.sin(x)
df_dx = ca.gradient(scalar_f, x)

df_fun = ca.Function("df_fun", [x], [scalar_f, df_dx])
f_value, derivative_value = df_fun(1.0)

print("f(x) =", scalar_f)
print("df/dx =", df_dx)
print("f(1) =", float(f_value))
print("df/dx at x=1 =", float(derivative_value))

In [ ]:
w = ca.SX.sym("w", 2, 1)
vector_g = ca.vertcat(w[0]**2 + w[1], ca.sin(w[0] - w[1]))
J = ca.jacobian(vector_g, w)

J_fun = ca.Function("J_fun", [w], [vector_g, J])
g_value, J_value = J_fun(ca.DM([1.0, 2.0]))

print("g(w) =")
print(vector_g)
print("Jacobian dg/dw =")
print(J)
print("g([1, 2]) =")
print(np.array(g_value, dtype=float))
print("Jacobian at [1, 2] =")
print(np.array(J_value, dtype=float))

In [ ]:
w = ca.SX.sym("w", 2, 1)
scalar_h = w[0]**2 + w[0] * w[1] + ca.sin(w[1])
H, grad_h = ca.hessian(scalar_h, w)

H_fun = ca.Function("H_fun", [w], [grad_h, H])
grad_value, H_value = H_fun(ca.DM([1.0, 0.5]))

print("h(w) =", scalar_h)
print("gradient of h =")
print(grad_h)
print("Hessian of h =")
print(H)
print("gradient at [1, 0.5] =")
print(np.array(grad_value, dtype=float))
print("Hessian at [1, 0.5] =")
print(np.array(H_value, dtype=float))

## 6. Important pitfall: NumPy vs CasADi symbolic math

Use NumPy for numeric arrays. Use CasADi operations for symbolic CasADi expressions.

For example, use `ca.sin(x)` rather than `np.sin(x)` when `x` is symbolic. The same idea applies to `ca.cos`, `ca.mtimes`, `ca.vertcat`, and other symbolic operations.

In [ ]:
x = ca.SX.sym("x")

safe = ca.sin(x)
print("Safe symbolic expression with ca.sin:", safe)

try:
    bad = np.sin(x)
    print("NumPy accepted this expression in this environment:", bad)
    print("Still prefer ca.sin for symbolic CasADi expressions.")
except Exception as e:
    print("This is why we use ca.sin for symbolic expressions:")
    print(type(e).__name__, e)

## 7. First optimization problem with Opti

`Opti` is CasADi's modeling interface for optimization problems.

Start with the scalar unconstrained problem

\begin{align}
\min_u \; (u + 2)^2.
\end{align}

The optimum is $u^* = -2$.

In [ ]:
opti = ca.Opti()
u = opti.variable()

opti.minimize((u + 2)**2)
opti.solver("ipopt", p_opts, s_opts)

sol = opti.solve()
u_star = sol.value(u)

print("Expected optimum: -2")
print("Computed optimum:", u_star)

## 8. Constrained scalar optimization

Now add the bound

\begin{align}
-1 \le u \le 1.
\end{align}

The unconstrained optimum is not feasible, so the constrained optimum moves to $u^* = -1$.

This is the same idea behind saturated LQR and constrained MPC: constraints can change the optimizer's decision.

In [ ]:
opti = ca.Opti()
u = opti.variable()

opti.minimize((u + 2)**2)
opti.subject_to(opti.bounded(-1, u, 1))
opti.solver("ipopt", p_opts, s_opts)

sol = opti.solve()
u_star = sol.value(u)

print("Expected constrained optimum: -1")
print("Computed constrained optimum:", u_star)

## 9. Optimization with parameters

Parameters are fixed data for one solve, but their values can change before the next solve.

Consider

\begin{align}
\min_u \; (u - p)^2
\quad \text{subject to} \quad -1 \le u \le 1.
\end{align}

In MPC, the current state is a parameter. We solve a similar optimization problem repeatedly with updated parameter values.

In [ ]:
opti = ca.Opti()
u = opti.variable()
p = opti.parameter()

opti.minimize((u - p)**2)
opti.subject_to(opti.bounded(-1, u, 1))
opti.solver("ipopt", p_opts, s_opts)

p_values = np.array([-2.0, -0.5, 0.5, 2.0])
u_values = []

for p_value in p_values:
    opti.set_value(p, p_value)
    sol = opti.solve()
    u_opt = sol.value(u)
    u_values.append(u_opt)
    opti.set_initial(u, u_opt)
    print(f"p = {p_value:4.1f} -> u* = {u_opt:5.2f}")

u_values = np.array(u_values)

fig, ax = plt.subplots()
ax.plot(p_values, u_values, "o-", label="optimal $u$")
ax.axhline(1.0, color="tab:red", linestyle="--", label="input bounds")
ax.axhline(-1.0, color="tab:red", linestyle="--")
ax.set_xlabel("parameter $p$")
ax.set_ylabel("optimal input $u^*$")
ax.set_title("Parameterized scalar optimization")
ax.legend(loc="best")
plt.show()

## 10. Tiny one-step optimal control problem

Now use the scalar dynamics

\begin{align}
x_{\text{next}} = x_0 + u
\end{align}

with cost

\begin{align}
x_0^2 + u^2 + x_{\text{next}}^2
\end{align}

and input constraint $-1 \le u \le 1$.

This is already a tiny MPC-like optimization problem: the current state is a parameter, and the optimizer chooses a constrained input.

In [ ]:
opti = ca.Opti()
u = opti.variable()
x0_param = opti.parameter()

x_next = x0_param + u
cost = x0_param**2 + u**2 + x_next**2

opti.minimize(cost)
opti.subject_to(opti.bounded(-1, u, 1))
opti.solver("ipopt", p_opts, s_opts)

x0_values = np.linspace(-4.0, 4.0, 17)
u_values = []

for x0_value in x0_values:
    opti.set_value(x0_param, x0_value)
    sol = opti.solve()
    u_opt = sol.value(u)
    u_values.append(u_opt)
    opti.set_initial(u, u_opt)

u_values = np.array(u_values)

fig, ax = plt.subplots()
ax.plot(x0_values, u_values, "o-", label="optimal $u$")
ax.axhline(1.0, color="tab:red", linestyle="--", label="input bounds")
ax.axhline(-1.0, color="tab:red", linestyle="--")
ax.set_xlabel("current state $x_0$")
ax.set_ylabel("optimal input $u^*$")
ax.set_title("One-step constrained optimal control")
ax.legend(loc="best")
plt.show()

## 11. Small finite-horizon problem

Now build a finite-horizon problem for the scalar system

\begin{align}
x_{k+1} = x_k + u_k,
\end{align}

with $Q=1$, $R=1$, horizon $N=5$, and $|u_k| \le 1$.

The variables are the predicted state sequence `X` and input sequence `U`. The parameter is the measured current state.

In [ ]:
Q = 1.0
R = 1.0
N = 5
u_limit = 1.0

opti = ca.Opti()
X = opti.variable(1, N + 1)
U = opti.variable(1, N)
x0_param = opti.parameter()

total_cost = 0
opti.subject_to(X[0, 0] == x0_param)

for k in range(N):
    opti.subject_to(X[0, k + 1] == X[0, k] + U[0, k])
    opti.subject_to(opti.bounded(-u_limit, U[0, k], u_limit))
    total_cost += Q * X[0, k]**2 + R * U[0, k]**2

total_cost += Q * X[0, N]**2

opti.minimize(total_cost)
opti.solver("ipopt", p_opts, s_opts)

opti.set_value(x0_param, 4.0)
opti.set_initial(X, np.linspace(4.0, 0.0, N + 1).reshape(1, N + 1))
opti.set_initial(U, -0.8 * np.ones((1, N)))

sol = opti.solve()
X_pred = sol.value(X).reshape(-1)
U_pred = sol.value(U).reshape(-1)

print("Predicted states:", X_pred)
print("Predicted inputs:", U_pred)

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=False)

axes[0].plot(np.arange(N + 1), X_pred, "o-")
axes[0].set_ylabel("state $x_k$")
axes[0].set_title("Finite-horizon prediction from $x_0 = 4$")

axes[1].step(np.arange(N), U_pred, where="post", label="input $u_k$")
axes[1].axhline(u_limit, color="tab:red", linestyle="--", label="input bounds")
axes[1].axhline(-u_limit, color="tab:red", linestyle="--")
axes[1].set_xlabel("prediction step $k$")
axes[1].set_ylabel("input $u_k$")
axes[1].legend(loc="best")

fig.tight_layout()
plt.show()

This is the structure of finite-horizon optimal control:

- variables for predicted states and inputs;
- a parameter for the current state;
- dynamics constraints;
- input constraints;
- a stage cost and terminal cost.

## 12. Receding-horizon MPC simulation

MPC solves a finite-horizon problem repeatedly, but applies only the first input.

At each simulation step:

1. set the current state parameter;
2. solve the finite-horizon problem;
3. apply the first input;
4. update the state;
5. repeat.

In [ ]:
x_current = 4.0
n_steps = 12

closed_loop_states = [x_current]
closed_loop_inputs = []

for sim_step in range(n_steps):
    opti.set_value(x0_param, x_current)
    sol = opti.solve()

    X_solution = sol.value(X)
    U_solution = sol.value(U)
    u_apply = float(U_solution[0, 0])

    closed_loop_inputs.append(u_apply)
    x_current = x_current + u_apply
    closed_loop_states.append(x_current)

    opti.set_initial(X, X_solution)
    opti.set_initial(U, U_solution)

closed_loop_states = np.array(closed_loop_states)
closed_loop_inputs = np.array(closed_loop_inputs)

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=False)

axes[0].plot(np.arange(n_steps + 1), closed_loop_states, "o-")
axes[0].axhline(0.0, color="0.25", linestyle="--")
axes[0].set_ylabel("state $x_k$")
axes[0].set_title("Closed-loop scalar MPC simulation")

axes[1].step(np.arange(n_steps), closed_loop_inputs, where="post", label="applied input")
axes[1].axhline(u_limit, color="tab:red", linestyle="--", label="input bounds")
axes[1].axhline(-u_limit, color="tab:red", linestyle="--")
axes[1].set_xlabel("time step $k$")
axes[1].set_ylabel("input $u_k$")
axes[1].legend(loc="best")

fig.tight_layout()
plt.show()

print("Final state:", closed_loop_states[-1])

## 13. Tiny 2D double-integrator MPC

The same pattern extends to vector states.

Use the discrete double integrator

\begin{align}
x_{k+1} = A x_k + B u_k,
\qquad
A = \begin{bmatrix} 1 & 1 \\ 0 & 1 \end{bmatrix},
\qquad
B = \begin{bmatrix} 0 \\ 1 \end{bmatrix}.
\end{align}

The state contains position and velocity. The input is acceleration. We constrain $|u_k| \le 1$ and, for this small example, $|\text{position}| \le 5$.

In [ ]:
A = np.array([[1.0, 1.0],
              [0.0, 1.0]])
B = np.array([[0.0],
              [1.0]])
Q = np.eye(2)
R = np.array([[1.0]])
A_cas = ca.DM(A)
B_cas = ca.DM(B)
Q_cas = ca.DM(Q)
R_cas = ca.DM(R)
N = 10
u_limit = 1.0
position_limit = 5.0

opti2 = ca.Opti()
X2 = opti2.variable(2, N + 1)
U2 = opti2.variable(1, N)
x0_2d = opti2.parameter(2, 1)

total_cost = 0
opti2.subject_to(X2[:, 0] == x0_2d)

for k in range(N):
    x_k = X2[:, k]
    u_k = U2[:, k]
    x_next = ca.mtimes(A_cas, x_k) + ca.mtimes(B_cas, u_k)

    opti2.subject_to(X2[:, k + 1] == x_next)
    opti2.subject_to(opti2.bounded(-u_limit, U2[0, k], u_limit))
    opti2.subject_to(opti2.bounded(-position_limit, X2[0, k], position_limit))

    total_cost += ca.mtimes([x_k.T, Q_cas, x_k]) + ca.mtimes([u_k.T, R_cas, u_k])

opti2.subject_to(opti2.bounded(-position_limit, X2[0, N], position_limit))
terminal_state = X2[:, N]
total_cost += ca.mtimes([terminal_state.T, Q_cas, terminal_state])

opti2.minimize(total_cost)
opti2.solver("ipopt", p_opts, s_opts)

opti2.set_value(x0_2d, np.array([[4.0], [0.0]]))
opti2.set_initial(X2, np.vstack((np.linspace(4.0, 0.0, N + 1), np.zeros(N + 1))))
opti2.set_initial(U2, np.zeros((1, N)))

sol2 = opti2.solve()
X2_pred = sol2.value(X2)
U2_pred = sol2.value(U2).reshape(-1)

fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=False)

axes[0].plot(np.arange(N + 1), X2_pred[0, :], "o-")
axes[0].axhline(position_limit, color="tab:red", linestyle="--", label="position bounds")
axes[0].axhline(-position_limit, color="tab:red", linestyle="--")
axes[0].set_ylabel("position")
axes[0].set_title("2D double-integrator MPC prediction")
axes[0].legend(loc="best")

axes[1].plot(np.arange(N + 1), X2_pred[1, :], "s-")
axes[1].axhline(0.0, color="0.25", linestyle="--")
axes[1].set_ylabel("velocity")

axes[2].step(np.arange(N), U2_pred, where="post", label="input")
axes[2].axhline(u_limit, color="tab:red", linestyle="--", label="input bounds")
axes[2].axhline(-u_limit, color="tab:red", linestyle="--")
axes[2].set_xlabel("prediction step $k$")
axes[2].set_ylabel("input")
axes[2].legend(loc="best")

fig.tight_layout()
plt.show()

The prediction is useful, but MPC is a feedback method. The final cell repeats the same 2D optimization with the current state as a changing parameter and applies only the first input.

In [ ]:
x_current = np.array([[4.0], [0.0]])
n_steps = 14

states_2d = [x_current.reshape(-1).copy()]
inputs_2d = []

for sim_step in range(n_steps):
    opti2.set_value(x0_2d, x_current)
    sol2 = opti2.solve()

    X2_solution = sol2.value(X2)
    U2_solution = sol2.value(U2)
    u_apply = float(U2_solution[0, 0])

    inputs_2d.append(u_apply)
    x_current = A @ x_current + B * u_apply
    states_2d.append(x_current.reshape(-1).copy())

    opti2.set_initial(X2, X2_solution)
    opti2.set_initial(U2, U2_solution)

states_2d = np.array(states_2d)
inputs_2d = np.array(inputs_2d)

fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=False)

axes[0].plot(np.arange(n_steps + 1), states_2d[:, 0], "o-")
axes[0].axhline(position_limit, color="tab:red", linestyle="--", label="position bounds")
axes[0].axhline(-position_limit, color="tab:red", linestyle="--")
axes[0].set_ylabel("position")
axes[0].set_title("Closed-loop 2D MPC simulation")
axes[0].legend(loc="best")

axes[1].plot(np.arange(n_steps + 1), states_2d[:, 1], "s-")
axes[1].axhline(0.0, color="0.25", linestyle="--")
axes[1].set_ylabel("velocity")

axes[2].step(np.arange(n_steps), inputs_2d, where="post", label="applied input")
axes[2].axhline(u_limit, color="tab:red", linestyle="--", label="input bounds")
axes[2].axhline(-u_limit, color="tab:red", linestyle="--")
axes[2].set_xlabel("time step $k$")
axes[2].set_ylabel("input")
axes[2].legend(loc="best")

fig.tight_layout()
plt.show()

print("Final 2D state:", states_2d[-1])

## 14. Summary

- CasADi builds symbolic expressions.
- CasADi can compute derivatives automatically.
- `Opti` formulates optimization problems.
- Parameters allow repeated solves with changing data.
- MPC uses the current state as a parameter.
- In this notebook, we built simple MPC problems directly and explicitly.
- In larger projects, the same idea may be hidden inside a helper class.